In [2]:
import pandas as pd
import numpy as np

In [3]:
import glob

orders_files = sorted(glob.glob("data/orders_2024_*.csv"))
print(orders_files)

df_orders = pd.concat(
    [pd.read_csv(f) for f in orders_files],
    ignore_index=True
)

df_orders["order_datetime"] = pd.to_datetime(df_orders["order_datetime"])

print(df_orders.shape)
print(df_orders["order_datetime"].min(), df_orders["order_datetime"].max())

['data/orders_2024_01.csv', 'data/orders_2024_03.csv']
(10000, 7)
2024-01-01 00:10:25 2024-03-31 23:34:00


In [4]:
print(df_orders["order_datetime"].dt.month.value_counts().sort_index())


order_datetime
1    5000
3    5000
Name: count, dtype: int64


In [8]:
df_orders["order_datetime_utc"] = df_orders["order_datetime"].dt.tz_localize("UTC")
df_orders["order_datetime_msk"] = df_orders["order_datetime_utc"].dt.tz_convert("Europe/Moscow")

print(df_orders[["order_datetime", "order_datetime_utc", "order_datetime_msk"]].head())

       order_datetime        order_datetime_utc        order_datetime_msk
0 2024-01-03 18:39:26 2024-01-03 18:39:26+00:00 2024-01-03 21:39:26+03:00
1 2024-01-03 17:45:43 2024-01-03 17:45:43+00:00 2024-01-03 20:45:43+03:00
2 2024-01-06 22:46:38 2024-01-06 22:46:38+00:00 2024-01-07 01:46:38+03:00
3 2024-01-08 21:03:51 2024-01-08 21:03:51+00:00 2024-01-09 00:03:51+03:00
4 2024-01-03 23:26:53 2024-01-03 23:26:53+00:00 2024-01-04 02:26:53+03:00


In [6]:
%pip install tzdata

In [7]:
import tzdata

In [9]:
orders_ts = df_orders.set_index("order_datetime_msk").sort_index()
orders_ts["revenue"] = orders_ts["price"] * orders_ts["quantity"]

daily_orders = orders_ts.resample("D").agg(
    orders_count=("order_id", "nunique"),
    revenue_sum=("revenue", "sum"),
)

print(daily_orders.head(10))
print(daily_orders.shape)

                           orders_count  revenue_sum
order_datetime_msk                                  
2024-01-01 00:00:00+03:00           141   3888229.53
2024-01-02 00:00:00+03:00           165   4242645.43
2024-01-03 00:00:00+03:00           169   4517004.59
2024-01-04 00:00:00+03:00           168   3624982.14
2024-01-05 00:00:00+03:00           181   4763831.59
2024-01-06 00:00:00+03:00           160   4450229.73
2024-01-07 00:00:00+03:00           158   3967384.29
2024-01-08 00:00:00+03:00           161   4201864.84
2024-01-09 00:00:00+03:00           175   4442862.12
2024-01-10 00:00:00+03:00           169   4249550.74
(92, 2)


In [10]:
print((daily_orders["orders_count"] == 0).sum())

28


In [11]:
daily_orders["revenue_rolling_7d"] = (
    daily_orders["revenue_sum"]
    .rolling(window=7, min_periods=1)
    .mean()
)

print(daily_orders[["revenue_sum", "revenue_rolling_7d"]].head(10))

                           revenue_sum  revenue_rolling_7d
order_datetime_msk                                        
2024-01-01 00:00:00+03:00   3888229.53        3.888230e+06
2024-01-02 00:00:00+03:00   4242645.43        4.065437e+06
2024-01-03 00:00:00+03:00   4517004.59        4.215960e+06
2024-01-04 00:00:00+03:00   3624982.14        4.068215e+06
2024-01-05 00:00:00+03:00   4763831.59        4.207339e+06
2024-01-06 00:00:00+03:00   4450229.73        4.247821e+06
2024-01-07 00:00:00+03:00   3967384.29        4.207758e+06
2024-01-08 00:00:00+03:00   4201864.84        4.252563e+06
2024-01-09 00:00:00+03:00   4442862.12        4.281166e+06
2024-01-10 00:00:00+03:00   4249550.74        4.242958e+06


In [12]:
best_day = daily_orders["revenue_rolling_7d"].idxmax()
best_value = daily_orders["revenue_rolling_7d"].max()

print("Jour du maximum :", best_day.date())
print("Valeur :", best_value)

Jour du maximum : 2024-01-19
Valeur : 4535556.378571429
